# 무드월 감정 분류 v0 학습
런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후, 위에서부터 셀을 하나씩 실행하세요.

## 1. GPU 확인
표 안에 Tesla T4가 보이면 성공

In [ ]:
!nvidia-smi

## 2. 라이브러리 설치

In [ ]:
!pip install -q transformers scikit-learn

## 3. 시드 데이터 업로드
실행하면 '파일 선택' 버튼이 나옵니다. `seed_data.csv`를 고르세요.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 4. 데이터 합치기 (시드 + UnSmile 부적절 문장)

In [ ]:
import pandas as pd

seed = pd.read_csv("seed_data.csv", encoding="utf-8-sig")

# UnSmile: clean==0 인 문장(혐오/악플)을 부적절로 사용, 짧은 문장만
url = "https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_train_v1.0.tsv"
un = pd.read_csv(url, sep="\t")
un = un[(un["clean"] == 0) & (un["문장"].str.len() <= 40)]
un = un.sample(n=40, random_state=42)
un_df = pd.DataFrame({"text": un["문장"].values, "label": "inappropriate"})

df = pd.concat([seed[["text", "label"]], un_df]).drop_duplicates("text").reset_index(drop=True)
print("전체 문장 수:", len(df))
print(df["label"].value_counts())

## 5. 학습 준비 (라벨, 데이터 분리, 모델 불러오기)

In [ ]:
import random, numpy as np, torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABELS = ["congrats", "joy", "cheer", "excited", "tired", "sad", "angry", "inappropriate"]
KO = {"congrats": "축하", "joy": "기쁨", "cheer": "응원", "excited": "설렘",
      "tired": "피곤", "sad": "슬픔", "angry": "화남", "inappropriate": "부적절"}
label2id = {l: i for i, l in enumerate(LABELS)}

random.seed(42); np.random.seed(42); torch.manual_seed(42)

# 테스트셋 15%는 학습에 안 씀 (나중에 재학습 전후 비교 기준)
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)
print("학습:", len(train_df), "/ 테스트:", len(test_df))

MODEL = "beomi/KcELECTRA-base-v2022"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(LABELS),
    id2label={i: KO[l] for i, l in enumerate(LABELS)},
    label2id={KO[l]: i for i, l in enumerate(LABELS)},
).to(device)
print("사용 장치:", device)

## 6. 학습 (T4 기준 2~3분)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def encode(frame):
    enc = tok(list(frame["text"]), padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    y = torch.tensor([label2id[l] for l in frame["label"]])
    return TensorDataset(enc["input_ids"], enc["attention_mask"], y)

train_loader = DataLoader(encode(train_df), batch_size=16, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
EPOCHS = 8

model.train()
for epoch in range(EPOCHS):
    total = 0
    for ids, mask, y in train_loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        loss = model(input_ids=ids, attention_mask=mask, labels=y).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  loss {total/len(train_loader):.4f}")

## 7. 평가 (정확도, 라벨별 F1, 헷갈린 라벨)
이 셀의 출력 화면을 캡처해 두세요. v0 결과 기록입니다.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

model.eval()
preds, inapp_probs = [], []
with torch.no_grad():
    for i in range(0, len(test_df), 32):
        batch = list(test_df["text"].iloc[i:i+32])
        enc = tok(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
        prob = torch.softmax(model(**enc).logits, -1)
        preds += prob.argmax(-1).tolist()
        inapp_probs += prob[:, label2id["inappropriate"]].tolist()

y_true = [label2id[l] for l in test_df["label"]]
print("정확도:", round(accuracy_score(y_true, preds), 3))
print(classification_report(y_true, preds, labels=list(range(len(LABELS))),
                            target_names=[KO[l] for l in LABELS], zero_division=0))

# 혼동 행렬: 행=정답, 열=예측
cm = pd.crosstab(pd.Series([KO[LABELS[t]] for t in y_true], name="정답"),
                 pd.Series([KO[LABELS[p]] for p in preds], name="예측"))
print(cm)

# 틀린 문장만 보기
wrong = test_df.assign(pred=[KO[LABELS[p]] for p in preds])
wrong = wrong[wrong["label"].map(KO) != wrong["pred"]]
print(wrong)

## 7-2. 부적절 보류 기준값 정하기
부적절 확률이 기준값 이상이면 전광판에 바로 띄우지 않고 관리자 확인으로 보냅니다. '잡은 비율'은 높을수록, '정상 글 잘못 보류'는 낮을수록 좋습니다.

In [ ]:
is_inapp = np.array([t == label2id["inappropriate"] for t in y_true])
p = np.array(inapp_probs)
for th in [0.1, 0.2, 0.3, 0.4, 0.5]:
    hold = p >= th
    recall = (hold & is_inapp).sum() / max(is_inapp.sum(), 1)
    false_hold = (hold & ~is_inapp).sum() / max((~is_inapp).sum(), 1)
    print(f"기준 {th}: 부적절 잡은 비율 {recall:.0%}, 정상 글 잘못 보류 {false_hold:.0%}")

## 8. 직접 문장 넣어보기

In [ ]:
def predict(text):
    enc = tok(text, truncation=True, max_length=64, return_tensors="pt").to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0]
    i = int(prob.argmax())
    return {"label": model.config.id2label[i], "score": round(float(prob[i]), 4)}

for s in ["지수야 생일 ㅊㅋ!!", "내일 축제 개기대됨", "밤샘 과제 너무 지친다", "학식 왜 맨날 이럼", "기말 다들 화이팅"]:
    print(s, "→", predict(s))

## 9. 모델 저장 → 구글 드라이브
권한 허용 창이 뜨면 허용하세요. 끝나면 드라이브 '내 드라이브'에 `moodwall_model.zip`과 `test_set_v0.csv`가 생깁니다.

In [ ]:
model.save_pretrained("moodwall_model")
tok.save_pretrained("moodwall_model")
test_df.to_csv("test_set_v0.csv", index=False, encoding="utf-8-sig")

from google.colab import drive
drive.mount("/content/drive")
!zip -r -q moodwall_model.zip moodwall_model
!cp moodwall_model.zip test_set_v0.csv /content/drive/MyDrive/
print("저장 완료")